# Laboratorio 10: Búsqueda por Similitud

Dataset: **Dry Bean** (UCI id=602), 13 611 granos con 16 atributos numéricos y 7 clases.

## Configuración y Carga de Datos

```sql
CREATE EXTENSION IF NOT EXISTS cube;

CREATE TABLE dry_beans (
    id SERIAL PRIMARY KEY,
    features_seq cube,
    features_idx cube
);

CREATE INDEX dry_beans_features_idx_gist ON dry_beans USING gist (features_idx);
```

> La extensión, la tabla y el índice se crean al hacer `docker compose up` (ver `init.sql`). Los datos se cargan desde la celda siguiente.

La fila `i` del DataFrame (0-based) corresponde a `id = i + 1` en `dry_beans`.

In [3]:
import os
import pandas as pd

DATA_PATH = "data/Dry_Bean_Dataset.xlsx"

if os.path.exists(DATA_PATH):
    df = pd.read_excel(DATA_PATH)
else:
    from ucimlrepo import fetch_ucirepo
    dry_bean = fetch_ucirepo(id=602)
    X = dry_bean.data.features
    y = dry_bean.data.targets
    df = pd.concat([X, y], axis=1)
    os.makedirs("data", exist_ok=True)
    df.to_csv(DATA_PATH, index=False)

FEATURES = list(df.columns[:16])
TARGET = df.columns[16]

print(f"Filas: {len(df)}, atributos: {len(FEATURES)}, clases: {df[TARGET].nunique()}")
df.head()

Filas: 13611, atributos: 16, clases: 7


,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [7]:
import psycopg2
from psycopg2.extras import execute_values


def connect_db():
    conn = psycopg2.connect(
        dbname="postgres", user="postgres",
        password="postgres", host="localhost", port=os.getenv("PGPORT", "5432")
    )
    return conn


conn = connect_db()
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM dry_beans;")
    if cur.fetchone()[0] == 0:
        # id = fila del CSV + 1, aunque la secuencia se haya usado antes
        cur.execute("SELECT setval(pg_get_serial_sequence('dry_beans', 'id'), 1, false);")
        rows = [(vec, vec) for vec in df[FEATURES].astype(float).values.tolist()]
        execute_values(cur,
                       "INSERT INTO dry_beans (features_seq, features_idx) VALUES %s;",
                       rows, template="(cube(%s::float8[]), cube(%s::float8[]))", page_size=1000)
        conn.commit()
        print(f"Insertadas {len(rows)} filas en dry_beans")
    else:
        print("dry_beans ya tiene datos, no se inserta nada")
    cur.execute("SELECT COUNT(*) FROM dry_beans;")
    print(f"Filas en dry_beans: {cur.fetchone()[0]}")
conn.close()

Insertadas 13611 filas en dry_beans
Filas en dry_beans: 13611


## P1a

## P1b

## P2

## P3: KNN

## P3: Escalabilidad

## P3: Dimensionalidad